# LangChain vs LangGraph — Complete Comparison

> **LangChain builds linear pipelines. LangGraph builds stateful, cyclical, agent-grade workflows. Understanding when to use which is the key skill for production AI systems.**

---

## Table of Contents

| # | Topic |
|---|---|
| 1 | What is LangChain? |
| 2 | What is LangGraph? |
| 3 | Core Architecture Differences |
| 4 | State Management |
| 5 | Control Flow — Linear vs Graph |
| 6 | Loops & Cycles |
| 7 | Memory & Persistence |
| 8 | Human-in-the-Loop |
| 9 | Tooling & Agents |
| 10 | When to Use Each |
| 11 | Side-by-Side Code Comparisons |
| 12 | Migration — LangChain → LangGraph |
| 13 | Full Hands-on: Same task, both frameworks |

---

## 1. What is LangChain?

**LangChain** is a framework for building LLM-powered applications using composable, chainable building blocks.

### Core Philosophy
> Compose LLM calls, prompts, retrievers, and tools into **linear pipelines** using the `|` pipe operator (LCEL).

### What LangChain Provides

| Component | Purpose |
|---|---|
| `ChatPromptTemplate` | Structure prompts with variables |
| `ChatAnthropic` / `ChatOpenAI` | LLM wrappers |
| `StrOutputParser` | Parse LLM output to string |
| `RunnableSequence` | Chain components with `|` pipe |
| `RunnableParallel` | Run components in parallel |
| `RunnableBranch` | Basic conditional branching |
| `VectorStoreRetriever` | Retrieve documents for RAG |
| Tools (`@tool`) | Annotate Python functions as LLM tools |

### LangChain Expression Language (LCEL)

```python
# The core pattern: chain components with |
chain = prompt | llm | output_parser
result = chain.invoke({"question": "What is AI?"})
```

```
LangChain execution:

  Input ──► [Prompt] ──► [LLM] ──► [Parser] ──► Output
                  (always left-to-right, no loops)
```

### LangChain is Excellent For
- Simple one-shot Q&A apps
- RAG (Retrieval-Augmented Generation) pipelines
- Text summarisation, classification, extraction
- Prototyping LLM workflows quickly

---

## 2. What is LangGraph?

**LangGraph** is a library built on top of LangChain that models AI workflows as **directed state graphs** — enabling loops, branching, memory, and human oversight.

### Core Philosophy
> Model your AI workflow as a **graph**: nodes do work, edges control flow, and a shared state object carries all data.

### What LangGraph Provides

| Component | Purpose |
|---|---|
| `StateGraph` | The graph container — holds nodes + edges |
| `TypedDict` state | Single shared memory object |
| `add_node(name, fn)` | Register a node (Python function) |
| `add_edge(a, b)` | Always-run transition from a → b |
| `add_conditional_edges` | Route based on state at runtime |
| `START` / `END` | Entry and exit constants |
| `MemorySaver` | Persist state across conversation turns |
| `interrupt_before` | Pause graph for human approval |

```python
# The core pattern: define a graph, compile, invoke
builder = StateGraph(MyState)
builder.add_node("step_a", fn_a)
builder.add_node("step_b", fn_b)
builder.add_edge(START, "step_a")
builder.add_edge("step_a", "step_b")
builder.add_edge("step_b", END)
graph = builder.compile()
result = graph.invoke({"input": "What is AI?"})
```

```
LangGraph execution:

  START ──► [step_a] ──► [step_b] ──► END
                │                      ▲
                └── condition? ────────┘  (can loop, branch, pause)
```

### LangGraph is Excellent For
- Multi-step agent loops (plan → act → observe → repeat)
- Tool-using agents that need to retry
- Workflows that require human approval steps
- Multi-turn conversations with persistent memory
- Complex branching workflows

---

## 3. Core Architecture Differences

### The Fundamental Difference

| Dimension | LangChain | LangGraph |
|---|---|---|
| **Execution model** | Linear pipeline (DAG, no cycles) | Directed state graph (cycles allowed) |
| **Data flow** | Output of step N → input of step N+1 | Shared `TypedDict` state passed through all nodes |
| **Control flow** | Fixed at build time | Dynamic — resolved at runtime from state |
| **Loops** | ❌ Not supported natively | ✅ First-class cycles via `add_edge` back-edges |
| **Branching** | Limited (`RunnableBranch`) | Full conditional routing via `add_conditional_edges` |
| **State persistence** | ❌ No built-in memory | ✅ `MemorySaver`, `SqliteSaver` by `thread_id` |
| **Human-in-the-loop** | ❌ Not supported | ✅ `interrupt_before` / `interrupt_after` |
| **Observability** | LangSmith tracing | LangSmith + per-node state inspection |
| **Complexity** | Low — great for simple tasks | Higher — built for production agents |
| **Learning curve** | Gentle | Steeper |

### Mental Model

```
LangChain mental model:
──────────────────────
  Think: UNIX pipes
  data | step1 | step2 | step3 → output
  Everything is a Runnable. Chain them.

LangGraph mental model:
───────────────────────
  Think: State machine / flowchart
  Nodes = functions that read/write shared state
  Edges = transitions between functions
  The graph decides what runs next based on state.
```

### Relationship Between the Two

```
┌─────────────────────────────────────────┐
│              LangGraph                  │
│  (graph orchestration + state control)  │
│                                         │
│   ┌─────────────────────────────────┐   │
│   │           LangChain             │   │
│   │  (LLM calls, tools, prompts,    │   │
│   │   retrievers, parsers)          │   │
│   └─────────────────────────────────┘   │
└─────────────────────────────────────────┘

LangGraph does not replace LangChain.
LangGraph USES LangChain components inside its nodes.
```

---

## 4. State Management

### LangChain — No Shared State

In LangChain, each step receives the **output of the previous step only**.
There is no central state object — data is piped step-by-step.

```python
# LangChain: each step gets the previous step's output
chain = prompt | llm | parser

#  Input dict
#      ↓
#  [prompt]  → formats template with input dict
#      ↓       output: formatted prompt string
#   [llm]    → sends prompt, returns AIMessage
#      ↓       output: AIMessage object
#  [parser]  → extracts .content from AIMessage
#      ↓       output: plain string
#   Result
```

**Problem:** If step 3 needs something from step 1, you must restructure the entire chain.

---

### LangGraph — Shared TypedDict State

In LangGraph, **one shared state object** travels through the entire graph.
Every node can read any field and update any field.

```python
from typing import TypedDict, Annotated
import operator

class WorkflowState(TypedDict):
    question:    str                                    # user's original question
    documents:   list[str]                              # retrieved docs
    answer:      str                                    # final answer
    messages:    Annotated[list, operator.add]          # appended, not replaced
    retry_count: int                                    # loop counter

# Every node reads from and writes to this same object
def retriever_node(state: WorkflowState) -> dict:
    docs = retrieve(state["question"])       # reads 'question'
    return {"documents": docs}               # writes 'documents'

def answer_node(state: WorkflowState) -> dict:
    # can read BOTH 'question' and 'documents' — set by different nodes
    answer = llm(state["question"], state["documents"])
    return {"answer": answer}
```

### State Update Rules

| Field type | Behaviour | Annotation |
|---|---|---|
| `str`, `int`, `dict` | **Replaced** by the node's return value | None needed |
| `list` (messages, logs) | **Appended** — new items added to existing list | `Annotated[list, operator.add]` |

```python
# Replace:
answer: str          # return {"answer": "new"} replaces old answer

# Append:
messages: Annotated[list, operator.add]   # return {"messages": [new_msg]} appends
```

---

## 5. Control Flow — Linear vs Graph

### LangChain Control Flow

LangChain offers three patterns:

```python
from langchain_core.runnables import RunnableSequence, RunnableParallel, RunnableBranch

# 1. Sequential (default) — steps run one after another
chain = step_a | step_b | step_c

# 2. Parallel — steps run at the same time
parallel = RunnableParallel(branch_a=step_a, branch_b=step_b)

# 3. Branch — basic conditional (limited)
branch = RunnableBranch(
    (lambda x: "math" in x["question"], math_chain),
    default_chain  # fallback
)
```

```
LangChain — Sequential:
  Input ──► [A] ──► [B] ──► [C] ──► Output

LangChain — Parallel:
  Input ──► [A] ──┐
             [B] ──┼──► Merged Output
             [C] ──┘

LangChain — Branch (basic):
  Input ──► condition? ──Yes──► [chain_a] ──► Output
                        └─No──► [chain_b] ──► Output
```

❌ **What LangChain cannot do:** loop back, re-enter a previous step, or change routing after the graph is built.

---

### LangGraph Control Flow

LangGraph supports all LangChain patterns **plus** loops and dynamic routing:

```python
# Simple edge — always run B after A
builder.add_edge("node_a", "node_b")

# Conditional edge — route based on state
def router(state: MyState) -> str:
    if state["needs_tool"]:
        return "tool_node"
    return "answer_node"

builder.add_conditional_edges("agent_node", router)

# Cycle / loop — go back to a previous node
builder.add_edge("tool_node", "agent_node")   # loops back!
```

```
LangGraph — Full control flow:

  START ──► [agent] ──── has tool call? ──Yes──► [tools] ──┐
                    └── No ──► END                          │
                    ▲__________________________________ ─────┘
                                  (cycle)
```

### Control Flow Comparison Table

| Pattern | LangChain | LangGraph |
|---|---|---|
| Sequential steps | ✅ `a \| b \| c` | ✅ `add_edge(a, b)` |
| Parallel steps | ✅ `RunnableParallel` | ✅ parallel edges |
| Conditional branch | ⚠️ `RunnableBranch` (limited) | ✅ `add_conditional_edges` (full) |
| Loops / cycles | ❌ | ✅ `add_edge(b, a)` |
| Dynamic routing at runtime | ❌ | ✅ routing function reads state |
| Pause mid-execution | ❌ | ✅ `interrupt_before` |

---

## 6. Loops & Cycles

This is the single most important difference between the two frameworks.

### Why Loops Matter for AI Agents

Real agent behaviour is inherently iterative:

```
1. Agent decides it needs to search
2. Agent calls search tool
3. Agent reads result — "not enough, need another search"
4. Agent calls search tool again
5. Agent reads result — "enough, compose answer"
6. Done
```

Steps 3-4 are a **loop**. LangChain cannot model this naturally.

---

### LangChain — Simulating Loops (Workaround)

```python
# In LangChain, you must write Python loops manually
# outside the chain — the chain itself cannot loop

result = {"question": user_input, "tool_results": []}
for _ in range(max_iterations):           # manual loop in Python
    result = agent_chain.invoke(result)
    if result["done"]:
        break
# This works but loses traceability, state management, and checkpointing
```

❌ Problems with this approach:
- Loop logic is **outside** the framework — hard to trace, debug, checkpoint
- No way to pause and resume mid-loop
- State management is manual

---

### LangGraph — Native Cycles

```python
# LangGraph — loop is declared as part of the graph

def should_continue(state: AgentState) -> str:
    """Routing function — decides whether to loop or stop."""
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"          # loop back through tools
    return END                  # done — exit graph

builder.add_conditional_edges("agent", should_continue)
builder.add_edge("tools", "agent")    # ← this creates the loop
```

```
LangGraph native agent loop:

  START
    ↓
  [agent_node]             ← LLM decides what to do
    │
    ├── tool_calls? ──Yes──► [tools_node]    ← execute tools
    │                             │
    │                        (loop back)
    │                             ↓
    │                        [agent_node]    ← LLM sees tool results
    │
    └── No tool calls ──► END
```

✅ Benefits:
- Loop is **inside** the graph — fully traced by LangSmith
- Can checkpoint state at any loop iteration
- Can interrupt before any iteration for human review

---

## 7. Memory & Persistence

### LangChain — Conversation Memory (Legacy)

LangChain has `ConversationBufferMemory` and similar classes, but these are:
- Tied to a specific chain instance
- Not persistent across process restarts
- Complex to integrate correctly — often leads to bugs
- Being phased out in favour of LangGraph

```python
# LangChain legacy memory — works but fragile
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(return_messages=True)
chain = ConversationChain(llm=llm, memory=memory)

chain.predict(input="Hi!")         # stored in memory
chain.predict(input="My name is Alice")  # memory grows
# ❌ Restart the process → memory is gone
```

---

### LangGraph — Checkpointer-Based Persistence

LangGraph persists the **entire state** after every node execution.
Conversations are resumed by `thread_id` — no state is ever lost.

```python
from langgraph.checkpoint.memory import MemorySaver

# Compile graph with a checkpointer
checkpointer = MemorySaver()           # in-memory (dev)
# checkpointer = SqliteSaver.from_conn_string("state.db")  # persistent (prod)

graph = builder.compile(checkpointer=checkpointer)

# Each conversation is identified by thread_id
config_alice = {"configurable": {"thread_id": "alice-session-1"}}
config_bob   = {"configurable": {"thread_id": "bob-session-1"}}

# Turn 1 — Alice
graph.invoke({"messages": [HumanMessage("Hi, I'm Alice")]}, config_alice)

# Turn 2 — Alice (state from Turn 1 is automatically loaded)
graph.invoke({"messages": [HumanMessage("What's my name?")]}, config_alice)
# ✅ LLM correctly answers: "Your name is Alice"
```

### Memory Comparison

| Feature | LangChain Memory | LangGraph Checkpointer |
|---|---|---|
| Persistence across restarts | ❌ (in-memory only) | ✅ `SqliteSaver`, custom stores |
| Multiple users / sessions | ❌ Manual | ✅ Per `thread_id` |
| State rollback / replay | ❌ | ✅ Built-in |
| Full state saved (not just messages) | ❌ | ✅ All state fields |
| Time-travel debugging | ❌ | ✅ Restore any past checkpoint |

---

## 8. Human-in-the-Loop

### LangChain — No Native Support

LangChain has no built-in way to pause execution for human input.
You must manually architect this outside the chain.

```python
# LangChain — manual human-in-the-loop (fragile)
result = step_1_chain.invoke(input)
human_approval = input(f"Approve this? {result['plan']} [y/n]: ")
if human_approval == "y":
    final = step_2_chain.invoke(result)
# ❌ Not resumable, not traceable, not checkpointable
```

---

### LangGraph — `interrupt_before` / `interrupt_after`

LangGraph can pause the graph **mid-execution** before any named node.
The state is checkpointed. The human reviews and resumes.

```python
# Compile with interrupt before a sensitive node
graph = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["execute_action"]   # pause before this node
)

config = {"configurable": {"thread_id": "review-1"}}

# Phase 1: Run up to the interrupt
result = graph.invoke({"task": "delete all logs"}, config)
# Graph pauses here — state is saved

# Human reviews the pending action
state = graph.get_state(config)
print("Pending action:", state.values["planned_action"])

# Human approves → resume the graph
graph.invoke(None, config)   # None = resume from checkpoint
# ✅ execute_action node now runs
```

```
LangGraph Human-in-the-Loop flow:

  START ──► [plan_node] ──► [review_node] ──✋ PAUSE
                                                │
                                          Human reviews
                                                │
                                          ▼ RESUME
                                       [execute_node] ──► END
```

---

## 9. Tooling & Agents

### LangChain — AgentExecutor (Legacy)

LangChain's original agent pattern uses `AgentExecutor` — a black-box loop.

```python
from langchain.agents import create_tool_calling_agent, AgentExecutor

tools = [search_tool, calculator_tool]
agent = create_tool_calling_agent(llm, tools, prompt)

executor = AgentExecutor(agent=agent, tools=tools)
result = executor.invoke({"input": "What is 25 * 4 + the population of France?"})
```

❌ Problems with `AgentExecutor`:
- Black box — hard to customise the loop logic
- No mid-execution pause
- Limited state visibility
- LangChain officially recommends migrating to LangGraph

---

### LangGraph — Tool Node Pattern

LangGraph provides a `ToolNode` that handles tool dispatch, and you control the loop explicitly.

```python
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_anthropic import ChatAnthropic

tools = [search_tool, calculator_tool]
llm_with_tools = ChatAnthropic(model="claude-opus-4-5").bind_tools(tools)

def agent_node(state: AgentState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

tool_node = ToolNode(tools)

builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)  # built-in routing
builder.add_edge("tools", "agent")   # loop back

graph = builder.compile()
```

✅ Advantages:
- You control what happens at each step
- Can add validation, logging, or custom logic between any steps
- Full LangSmith traceability per node
- Loop can be interrupted, checkpointed, replayed

### Agent Comparison

| Feature | LangChain `AgentExecutor` | LangGraph `ToolNode` |
|---|---|---|
| Custom loop logic | ❌ Black box | ✅ Full control |
| Interrupt mid-loop | ❌ | ✅ `interrupt_before` |
| State inspection per step | ❌ | ✅ |
| Add custom nodes between tool calls | ❌ | ✅ |
| Recommended by LangChain team | ❌ Deprecated | ✅ Recommended |

---

## 10. When to Use Each

### Decision Framework

```
START: What do I need to build?
         │
         ├── One-shot call (Q&A, summarise, classify)?
         │         └──► Use LangChain LCEL
         │
         ├── RAG pipeline (retrieve → generate)?
         │         └──► Use LangChain (or LangGraph for complex RAG)
         │
         ├── Agent that uses tools and may loop?
         │         └──► Use LangGraph
         │
         ├── Multi-turn conversation with memory?
         │         └──► Use LangGraph + checkpointer
         │
         ├── Workflow requiring human approval?
         │         └──► Use LangGraph + interrupt_before
         │
         └── Production agent with retry logic?
                   └──► Use LangGraph
```

### Use LangChain When

| Scenario | Example |
|---|---|
| Simple one-shot LLM call | "Summarise this document" |
| Fixed RAG pipeline | Retrieve 3 docs → generate answer |
| Prompt engineering / testing | Experimenting with prompt templates |
| Simple data extraction | Extract JSON from unstructured text |
| Basic classification | Categorise support tickets |

### Use LangGraph When

| Scenario | Example |
|---|---|
| Tool-using agent with loops | Research agent that searches until confident |
| Multi-turn chatbot with memory | Customer service bot remembering context |
| Human approval workflows | "Draft email → human reviews → send" |
| Retry / validation loops | Generate → validate → regenerate if invalid |
| Complex branching | Route to specialist agents based on intent |
| Long-running workflows | Async tasks that can be paused and resumed |

---

## 11. Side-by-Side Code Comparisons

### Comparison A — Simple Q&A

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# COMPARISON A: Simple Q&A
# ─────────────────────────────────────────────────────────────────────────────

# ── LangChain version ─────────────────────────────────────────────────────────
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatAnthropic(model="claude-opus-4-5")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human",  "{question}")
])

lc_chain = prompt | llm | StrOutputParser()

lc_result = lc_chain.invoke({"question": "What is the capital of France?"})
print("LangChain result:", lc_result)

# ── LangGraph version ─────────────────────────────────────────────────────────
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage

class QAState(TypedDict):
    question: str
    answer:   str

def qa_node(state: QAState) -> dict:
    messages = [
        SystemMessage("You are a helpful assistant."),
        HumanMessage(state["question"])
    ]
    response = llm.invoke(messages)
    return {"answer": response.content}

builder = StateGraph(QAState)
builder.add_node("qa", qa_node)
builder.add_edge(START, "qa")
builder.add_edge("qa",  END)
lg_graph  = builder.compile()

lg_result = lg_graph.invoke({"question": "What is the capital of France?", "answer": ""})
print("LangGraph result:", lg_result["answer"])

# ── Verdict ────────────────────────────────────────────────────────────────────
print("\n" + "─"*60)
print("VERDICT: For simple Q&A → LangChain is simpler and more concise.")
print("LangGraph adds boilerplate for no extra benefit here.")

### Comparison B — RAG (Retrieval-Augmented Generation)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# COMPARISON B: RAG Pipeline
# Simulated retrieval (no vector DB needed to run this cell)
# ─────────────────────────────────────────────────────────────────────────────

from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

llm = ChatAnthropic(model="claude-opus-4-5")

# Simulated retriever (replace with real vector store in production)
def fake_retriever(question: str) -> list[str]:
    return [
        "LangGraph was introduced by LangChain in 2024.",
        "LangGraph builds on top of LangChain to support cyclic workflows.",
        "LangGraph uses StateGraph and nodes to orchestrate LLM agents."
    ]

# ── LangChain RAG ─────────────────────────────────────────────────────────────
rag_prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context below.

Context:
{context}

Question: {question}
""")

def format_docs(docs: list[str]) -> str:
    return "\n".join(f"- {d}" for d in docs)

lc_rag_chain = (
    {
        "context":  lambda x: format_docs(fake_retriever(x["question"])),
        "question": RunnablePassthrough() | (lambda x: x["question"])
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

lc_rag_result = lc_rag_chain.invoke({"question": "What is LangGraph?"})
print("LangChain RAG:", lc_rag_result)

# ── LangGraph RAG ─────────────────────────────────────────────────────────────
class RAGState(TypedDict):
    question:  str
    documents: list[str]
    answer:    str

def retrieve_node(state: RAGState) -> dict:
    docs = fake_retriever(state["question"])
    print(f"[retrieve_node] found {len(docs)} documents")
    return {"documents": docs}

def generate_node(state: RAGState) -> dict:
    context = "\n".join(f"- {d}" for d in state["documents"])
    prompt_text = f"""Answer the question using only the context below.

Context:
{context}

Question: {state['question']}"""
    response = llm.invoke(prompt_text)
    print(f"[generate_node] answer generated")
    return {"answer": response.content}

rag_builder = StateGraph(RAGState)
rag_builder.add_node("retrieve", retrieve_node)
rag_builder.add_node("generate", generate_node)
rag_builder.add_edge(START,      "retrieve")
rag_builder.add_edge("retrieve", "generate")
rag_builder.add_edge("generate", END)
rag_graph = rag_builder.compile()

lg_rag_result = rag_graph.invoke({"question": "What is LangGraph?", "documents": [], "answer": ""})
print("LangGraph RAG:", lg_rag_result["answer"])

print("\n" + "─"*60)
print("VERDICT: For basic RAG → LangChain is slightly simpler.")
print("LangGraph shines when RAG needs loops (re-retrieve if answer is poor).")

### Comparison C — Looping Agent (Where LangGraph Dominates)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# COMPARISON C: Looping Agent
# Task: validate output quality — retry up to 3 times if quality is poor
# ─────────────────────────────────────────────────────────────────────────────

from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model="claude-opus-4-5")

# ── LangChain "approach" (must use manual loop outside chain) ──────────────────
print("LangChain approach (manual loop — not inside the chain):")

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

generate_prompt = ChatPromptTemplate.from_template(
    "Write a 1-sentence explanation of {topic}. Be clear and concise."
)
lc_generate_chain = generate_prompt | llm | StrOutputParser()

topic = "quantum entanglement"
lc_answer = ""
for attempt in range(3):                     #  ← loop is OUTSIDE the chain
    lc_answer = lc_generate_chain.invoke({"topic": topic})
    if len(lc_answer) > 50:                  # crude quality check
        print(f"  Attempt {attempt+1}: PASS — {lc_answer[:60]}...")
        break
    else:
        print(f"  Attempt {attempt+1}: FAIL — answer too short, retrying")
print(f"  ❌ Loop is manual Python, not inside framework — no checkpointing\n")

# ── LangGraph approach (loop is INSIDE the graph) ─────────────────────────────
print("LangGraph approach (loop is declared inside the graph):")

class QualityLoopState(TypedDict):
    topic:    str
    answer:   str
    attempts: int
    log:      Annotated[list[str], operator.add]

def generate_answer(state: QualityLoopState) -> dict:
    prompt = f"Write a 1-sentence explanation of {state['topic']}. Be clear and concise."
    response = llm.invoke(prompt)
    attempt_num = state["attempts"] + 1
    print(f"  [generate] Attempt {attempt_num}")
    return {
        "answer":   response.content,
        "attempts": attempt_num,
        "log":      [f"Attempt {attempt_num}: generated answer"]
    }

def validate_answer(state: QualityLoopState) -> dict:
    is_good = len(state["answer"]) > 50
    quality  = "PASS" if is_good else "FAIL"
    print(f"  [validate] {quality} — length={len(state['answer'])}")
    return {"log": [f"Attempt {state['attempts']}: {quality}"]}

def should_retry(state: QualityLoopState) -> str:
    """Routing function — loop back or exit."""
    if len(state["answer"]) <= 50 and state["attempts"] < 3:
        return "generate"     # loop back
    return END                  # exit

loop_builder = StateGraph(QualityLoopState)
loop_builder.add_node("generate", generate_answer)
loop_builder.add_node("validate", validate_answer)
loop_builder.add_edge(START,        "generate")
loop_builder.add_edge("generate",   "validate")
loop_builder.add_conditional_edges("validate", should_retry)  # ← loop declared here
loop_graph = loop_builder.compile()

lg_loop_result = loop_graph.invoke({
    "topic": "quantum entanglement",
    "answer": "", "attempts": 0, "log": []
})
print(f"  ✅ Loop is inside graph — fully traced, checkpointable")
print(f"  Total attempts: {lg_loop_result['attempts']}")
print(f"  Log: {lg_loop_result['log']}")
print(f"  Answer: {lg_loop_result['answer'][:80]}...")

---

## 12. Migration — LangChain → LangGraph

### Pattern Map

| LangChain Pattern | LangGraph Equivalent |
|---|---|
| `prompt \| llm \| parser` | `add_node("llm_node", fn)` + edges |
| `RunnableParallel(a=x, b=y)` | Two nodes with edges from the same source |
| `RunnableBranch(cond, chain_a, chain_b)` | `add_conditional_edges(node, routing_fn)` |
| `ConversationBufferMemory` | `Annotated[list, operator.add]` + `MemorySaver` |
| `AgentExecutor` | `ToolNode` + `add_edge(tools, agent)` loop |
| `chain.invoke(input)` | `graph.invoke(state)` |
| `chain.stream(input)` | `graph.stream(state)` |

### Step-by-Step Migration Example

```python
# ── BEFORE: LangChain chain ─────────────────────────────────────
chain = prompt | llm | parser
result = chain.invoke({"question": user_input})


# ── AFTER: LangGraph equivalent ─────────────────────────────────
class MyState(TypedDict):
    question: str
    answer:   str

def llm_node(state: MyState) -> dict:
    formatted = prompt.format_messages(question=state["question"])
    response  = llm.invoke(formatted)
    answer    = parser.invoke(response)
    return {"answer": answer}

builder = StateGraph(MyState)
builder.add_node("llm", llm_node)
builder.add_edge(START, "llm")
builder.add_edge("llm", END)
graph  = builder.compile()
result = graph.invoke({"question": user_input, "answer": ""})
```

### Note on Using LangChain Inside LangGraph

> LangGraph nodes are plain Python functions. You can freely use **LangChain chains inside a node**.

```python
# ✅ Perfectly valid — use a LangChain chain INSIDE a LangGraph node
lc_summarise_chain = summarise_prompt | llm | StrOutputParser()

def summarise_node(state: DocState) -> dict:
    summary = lc_summarise_chain.invoke({"text": state["document"]})
    return {"summary": summary}
```

This is the recommended hybrid pattern — use LangChain for component-level logic, LangGraph for workflow orchestration.

---

## 13. Full Hands-on: Same Task, Both Frameworks

**Task:** Build a **topic classifier** that:
1. Classifies the user's question as `math`, `science`, or `general`
2. Routes to an appropriate specialist prompt
3. Returns a tailored answer

We will build this **twice** — once with LangChain, once with LangGraph.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FULL HANDS-ON — Part 1: LangChain Implementation
# ─────────────────────────────────────────────────────────────────────────────

from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch, RunnableLambda

llm = ChatAnthropic(model="claude-opus-4-5")

# Step 1 — Classify the question
classify_prompt = ChatPromptTemplate.from_template("""
Classify this question into exactly one category: math, science, or general.
Reply with ONLY the category word.

Question: {question}
""")
classify_chain = classify_prompt | llm | StrOutputParser()

# Step 2 — Specialist answer chains
math_prompt     = ChatPromptTemplate.from_template("You are a maths tutor. Answer: {question}")
science_prompt  = ChatPromptTemplate.from_template("You are a science teacher. Answer: {question}")
general_prompt  = ChatPromptTemplate.from_template("You are a helpful assistant. Answer: {question}")

math_chain    = math_prompt    | llm | StrOutputParser()
science_chain = science_prompt | llm | StrOutputParser()
general_chain = general_prompt | llm | StrOutputParser()

# Step 3 — Wire together with RunnableBranch
def route_lc(inputs: dict) -> dict:
    """Classify and attach category to inputs."""
    category = classify_chain.invoke({"question": inputs["question"]}).strip().lower()
    return {"question": inputs["question"], "category": category}

lc_router = RunnableBranch(
    (lambda x: x["category"] == "math",    math_chain),
    (lambda x: x["category"] == "science", science_chain),
    general_chain   # fallback
)

lc_full_chain = RunnableLambda(route_lc) | lc_router

# Test it
test_questions = [
    "What is the derivative of x squared?",
    "Why is the sky blue?",
    "Who wrote Hamlet?"
]

print("═" * 60)
print("LANGCHAIN RESULTS")
print("═" * 60)
for q in test_questions:
    answer = lc_full_chain.invoke({"question": q})
    print(f"Q: {q}")
    print(f"A: {answer[:120]}...\n")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FULL HANDS-ON — Part 2: LangGraph Implementation
# ─────────────────────────────────────────────────────────────────────────────

from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatAnthropic(model="claude-opus-4-5")

# ── State ─────────────────────────────────────────────────────────────────────
class ClassifierState(TypedDict):
    question: str
    category: str      # filled by classify_node
    answer:   str      # filled by specialist node
    log:      Annotated[list[str], operator.add]

# ── Nodes ─────────────────────────────────────────────────────────────────────
def classify_node(state: ClassifierState) -> dict:
    """Node 1 — classify the question."""
    classify_prompt = ChatPromptTemplate.from_template("""
Classify this question into exactly one category: math, science, or general.
Reply with ONLY the category word.

Question: {question}
""")
    chain    = classify_prompt | llm | StrOutputParser()
    category = chain.invoke({"question": state["question"]}).strip().lower()
    print(f"[classify_node] → '{category}'")
    return {"category": category, "log": [f"classified as: {category}"]}

def math_node(state: ClassifierState) -> dict:
    """Specialist node for maths questions."""
    prompt   = f"You are a maths tutor. Answer clearly: {state['question']}"
    response = llm.invoke(prompt)
    print(f"[math_node] answered")
    return {"answer": response.content, "log": ["routed to math_node"]}

def science_node(state: ClassifierState) -> dict:
    """Specialist node for science questions."""
    prompt   = f"You are a science teacher. Answer clearly: {state['question']}"
    response = llm.invoke(prompt)
    print(f"[science_node] answered")
    return {"answer": response.content, "log": ["routed to science_node"]}

def general_node(state: ClassifierState) -> dict:
    """General-purpose node for other questions."""
    prompt   = f"You are a helpful assistant. Answer clearly: {state['question']}"
    response = llm.invoke(prompt)
    print(f"[general_node] answered")
    return {"answer": response.content, "log": ["routed to general_node"]}

# ── Routing function ──────────────────────────────────────────────────────────
def route_by_category(state: ClassifierState) -> str:
    """Reads state['category'] and returns the next node name."""
    routing_map = {
        "math":    "math_node",
        "science": "science_node",
        "general": "general_node"
    }
    return routing_map.get(state["category"], "general_node")

# ── Build graph ───────────────────────────────────────────────────────────────
clf_builder = StateGraph(ClassifierState)

clf_builder.add_node("classify", classify_node)
clf_builder.add_node("math_node",    math_node)
clf_builder.add_node("science_node", science_node)
clf_builder.add_node("general_node", general_node)

clf_builder.add_edge(START,       "classify")
clf_builder.add_conditional_edges(
    "classify",
    route_by_category,
    {
        "math_node":    "math_node",
        "science_node": "science_node",
        "general_node": "general_node"
    }
)
clf_builder.add_edge("math_node",    END)
clf_builder.add_edge("science_node", END)
clf_builder.add_edge("general_node", END)

clf_graph = clf_builder.compile()
print("✅ Classifier graph compiled\n")

# ── Test it ───────────────────────────────────────────────────────────────────
test_questions = [
    "What is the derivative of x squared?",
    "Why is the sky blue?",
    "Who wrote Hamlet?"
]

print("═" * 60)
print("LANGGRAPH RESULTS")
print("═" * 60)
for q in test_questions:
    print(f"\n{'─'*55}")
    print(f"Q: {q}")
    result = clf_graph.invoke({
        "question": q, "category": "", "answer": "", "log": []
    })
    print(f"Category: {result['category']}")
    print(f"Log:      {result['log']}")
    print(f"A: {result['answer'][:120]}...")

---

## Summary: LangChain vs LangGraph at a Glance

```
┌─────────────────────────────────────────────────────────────────────┐
│                                                                     │
│   LANGCHAIN                        LANGGRAPH                        │
│   ─────────                        ─────────                        │
│   Linear pipeline                  State graph (cycles allowed)     │
│   pipe operator  |                 add_edge / add_conditional_edges  │
│   Output → next step               Shared TypedDict state           │
│   No loops                         Native cycles                    │
│   No persistence                   MemorySaver / SqliteSaver        │
│   No human pause                   interrupt_before / after         │
│   Great for simple tasks           Great for agents & workflows     │
│                                                                     │
│   Both use LangSmith for observability                              │
│   LangGraph USES LangChain components inside nodes                 │
│   They are complementary — not competitors                         │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### The Golden Rule

| Task complexity | Framework |
|---|---|
| Single LLM call | LangChain |
| Fixed multi-step pipeline | LangChain |
| Agent with tools | LangGraph |
| Multi-turn memory | LangGraph |
| Complex workflow with branching, loops, human oversight | LangGraph |

> **Start with LangChain for simplicity. Reach for LangGraph when you need control.**